In [1]:
# Import the files
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# Import the data

inflation_df = pd.read_excel(
    Path("..", "teams", "TEAM00", "3", "bwf_inputs-TEAM00-stage_3", "economy", "economy-dynamic_properties-stage_3.xlsx"),
    sheet_name='inflation',
    index_col='timestamp',
    parse_dates=True
)

ele_prices_df = pd.read_excel(
    Path("..", "teams", "TEAM00", "3", "bwf_inputs-TEAM00-stage_3", "energy", "energy_system-dynamic_properties-stage_3.xlsx"),
    sheet_name='electricity_price-unit_cost',
    index_col='timestamp',
    parse_dates=True
)

ef_pipes_df = pd.read_excel(
    Path("..", "teams", "TEAM00", "3", "bwf_inputs-TEAM00-stage_3", "pipes", "pipe_options-dynamic_properties-stage_3.xlsx"),
    sheet_name='new_pipe-emissions_factor',
    index_col='timestamp',
    parse_dates=True
)
ef_grid_df = pd.read_excel(
    Path("..", "teams", "TEAM00", "3", "bwf_inputs-TEAM00-stage_3", "energy", "energy_system-dynamic_properties-stage_3.xlsx"),
    sheet_name='grid_emission_factor',
    index_col='timestamp',
    parse_dates=True
)

population_df = pd.read_excel(
    Path("..", "teams", "TEAM00", "3", "bwf_inputs-TEAM00-stage_3", "jurisdictions", "municipalities-dynamic_properties-stage_3.xlsx"),
    sheet_name='population',
    index_col='timestamp',
    parse_dates=True
)

In [3]:
# process the data

# normalise population
population_norm_df = population_df / population_df.max()

# join the emission factors
ef_df = pd.concat(
    [
        ef_pipes_df[['PI01', 'PI10']],
        ef_grid_df
    ],
    axis=1,
    join='outer',
    sort=True
)
# rename and normalize
ef_df.columns = ['PVC', 'steel', 'grid']
ef_df = ef_df / ef_df.max()

In [4]:
# Create the plot
WF_BLUE = '#3059D1'
WF_GREEN = '#009344'
WF_BLUE_FILL = 'rgba(48, 89, 209, 0.2)'
WF_GREEN_FILL = 'rgba(0, 147, 68, 0.5)'
WF_ORANGE = '#E8960C'
WF_RED = '#D63434'

fig_bwf_inputs = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Inflation',
        'Electricity Price',
        'Normalised Emission Factors',
        'Normalised Population'
    ),
    horizontal_spacing=0.1,
    vertical_spacing=0.12
)
fig_bwf_inputs.update_annotations(font=dict(size=26)) # Increase subplot subtitles

# Top left: Inflation
fig_bwf_inputs.add_trace(
    go.Scatter(
        x=inflation_df.index,
        y=inflation_df['NL0000'],
        mode='lines',
        name='Inflation (NL)',
        line=dict(color=WF_BLUE, width=2),
        hoverinfo='x+y+name'
    ),
    row=1, col=1
)
# Top right: Ele price
fig_bwf_inputs.add_trace(
    go.Scatter(
        x=ele_prices_df.index,
        y=ele_prices_df['NL0000'],
        mode='lines',
        name='Electricity price (NL)',
        line=dict(color=WF_ORANGE, width=2),
        hoverinfo='x+y+name'
    ),
    row=1, col=2
)
# Bottom left: Emission factors (3 lines)
dash_styles = ["dash", "dot", "solid"]
for i, col in enumerate(ef_df.columns):
    data = ef_df[col].dropna()
    fig_bwf_inputs.add_trace(
        go.Scatter(
            x=data.index,
            y=data.values,
            mode='lines',
            name=f'Emission factor ({col})(NL)',
            line=dict(color=WF_GREEN, width=2, dash=dash_styles[i]),
            hoverinfo='x+y+name'
        ),
        row=2, col=1
    )
# Bottom right: Population (n light lines and one full)

for col in population_norm_df.columns:
    fig_bwf_inputs.add_trace(
        go.Scatter(
            x=population_norm_df.index,
            y=population_norm_df[col],
            mode='lines',
            name=f'Population ({col})',
            showlegend=col == 'NL0000',
            line=dict(
                color=WF_RED if col == 'NL0000' else WF_BLUE_FILL,
                width=4 if col == 'NL0000' else 2,
            ),
            hoverinfo='x+y+name'
        ),
        row=2, col=2
    )

fig_bwf_inputs.add_annotation(
    x=population_df['NL0000'].idxmax(),
    y=1,
    text=f"{population_df['NL0000'].max():,.0f}".replace(',',"\u2009"),
    xanchor="right",
    showarrow=True,
    arrowhead=2,
    ax=0,
    ay=-10,
    font=dict(color="black", size=18),
    row=2, col=2
)

fig_bwf_inputs.update_layout(
    height=800,
    width=1600,
    title_text='',
    showlegend=True,
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        font=dict(size=20),
        title_font=dict(size=22),
    ),
)
axes_style=dict(
    ticks='outside',
    tickcolor='lightgray',
    tickwidth=1.5,
    showgrid=True,
    gridcolor='lightgray',
    gridwidth=1.2,
    zeroline=True,
    zerolinecolor='gray',
    zerolinewidth=2,
    title_font=dict(size=22),
    tickfont=dict(size=18),
)
xaxes_ticks=dict(
    tickvals=list(pd.date_range(start="2000-01-01", end="2100-01-10", freq="25YS")),
    ticktext=[str(y) for y in range(2000, 2101, 25)],
    range=[pd.to_datetime("1999-01-01"), pd.to_datetime("2101-01-01")]
)

fig_bwf_inputs.update_xaxes(title_text='', **axes_style, **xaxes_ticks, row=1, col=1)
fig_bwf_inputs.update_xaxes(title_text='', **axes_style, **xaxes_ticks, row=1, col=2)
fig_bwf_inputs.update_xaxes(title_text='Year', **axes_style, **xaxes_ticks, row=2, col=1)
fig_bwf_inputs.update_xaxes(title_text='Year', **axes_style, **xaxes_ticks, row=2, col=2)
fig_bwf_inputs.update_yaxes(title_text='Value [%]', **axes_style, row=1, col=1)
fig_bwf_inputs.update_yaxes(title_text='Cost [€/kWh]', **axes_style, row=1, col=2)
fig_bwf_inputs.update_yaxes(title_text='Ratio [-]', **axes_style, row=2, col=1)
fig_bwf_inputs.update_yaxes(title_text='Ratio [-]', **axes_style, range=[0, 1.05], row=2, col=2)

fig_bwf_inputs.show()

In [5]:
# Count participant masterplan files format
STAGES = [1, 2, 3]

n_json_ms = 0
n_yaml_ms = 0
n_xlsx_ms = 0

for team_path in sorted(Path("..", "teams").iterdir()):
    if not team_path.is_dir():
        continue
    
    team_id = team_path.name.split('-')[0]

    if team_id == 'TEAM00':
        continue
    
    for stage in STAGES:
        stage_dir = team_path / f"{stage}"
        results_dir = stage_dir / f"bwf_results-{team_id}-stage_{stage}"
        
        if not results_dir.exists():
            break  # If a stage is missing, skip the rest for this team

        ms_file = None
        ms_ext = None
        for f in stage_dir.iterdir():
            if f.is_file() and str(f.stem).startswith('masterplan-'):
                ms_file = f
                ms_ext = f.suffix
                break
        
        if ms_file is None:
            break

        if ms_ext == '.yaml' or ms_ext == '.yml':
            n_yaml_ms += 1

        elif ms_ext == '.json':
            n_json_ms += 1 

        elif ms_ext == '.xlsx':
            n_xlsx_ms += 1

        else:
            raise ValueError(f"Unkonwn file {ms_ext}")

print("Participants submitted a file in the following format this many times:")
print(f"- JSON {n_json_ms}")
print(f"- YAML {n_yaml_ms}")
print(f"- XLSX {n_xlsx_ms}")

Participants submitted a file in the following format this many times:
- JSON 2
- YAML 29
- XLSX 4


In [6]:
from water_futures_battle.masterplan import parse_masterplan

n_custom_ba = 0
n_pop_ba = 0
n_invpop_ba = 0
n_inc_ba = 0
n_invinc_ba = 0
n_ba = 0

n_custom_nrw = 0
n_class_nrw = 0
n_pop_nrw = 0
n_nrw = 0

n_custom_padj = 0
n_inf_padj = 0
n_padj = 0
n_low_padj = 0

for team_path in sorted(Path("..", "teams").iterdir()):
    if not team_path.is_dir():
        continue
    
    team_id = team_path.name.split('-')[0]

    if team_id == 'TEAM00':
        continue
    
    for stage in STAGES:
        stage_dir = team_path / f"{stage}"
        results_dir = stage_dir / f"bwf_results-{team_id}-stage_{stage}"
        
        if not results_dir.exists():
            break  # If a stage is missing, skip the rest for this team

        ms = None

        for f in stage_dir.iterdir():
            if f.is_file() and str(f.stem).startswith('masterplan-'):
                ms = parse_masterplan(f)
                break

        if ms is None: 
            continue

        for year in range(2000+25*stage, 2000+25*(stage+1)):
            policies = ms.national_policies(year=year)
            inflation = float(inflation_df.at[pd.Timestamp(year=year, month=1, day=1), "NL0000"])/100

            if policies['budget_allocation']['policy'] == 'custom':
                n_custom_ba += 1
            elif policies['budget_allocation']['policy'] == 'by_population':
                n_pop_ba += 1
            elif policies['budget_allocation']['policy'] == 'by_inverse_population':
                n_invpop_ba += 1
            elif policies['budget_allocation']['policy'] == 'by_income':
                n_inc_ba += 1
            elif policies['budget_allocation']['policy'] == 'by_inverse_income':
                n_invinc_ba += 1
            else:
                raise ValueError(f"Uknown {policies['budget_allocation']['policy']}")
            n_ba += 1

            for water_utility in [f"WU{x:02d}" for x in range(1, 11)]:
                policies = ms.water_utility_policies(
                    water_utility=water_utility,
                    year=year
                )

                if policies['nrw_mitigation']['policy'] == 'custom':
                    n_custom_nrw += 1
                elif policies['nrw_mitigation']['policy'] == 'by_population':
                    n_pop_nrw += 1
                elif policies['nrw_mitigation']['policy'] == 'by_nrw_class':
                    n_class_nrw += 1
                else:
                    raise ValueError(f"Uknown {policies['nrw_mitigation']['policy']}")
                
                n_nrw += 1

                if policies['pricing_adjustment']['policy'] == 'custom':
                    n_custom_padj += 1

                    if policies['pricing_adjustment']['policy_args']['fixed_component'] < inflation:
                        n_low_padj += 1/3
                    
                    if policies['pricing_adjustment']['policy_args']['variable_component'] < inflation:
                        n_low_padj += 1/3
                        
                    if policies['pricing_adjustment']['policy_args']['selling_price'] < inflation:
                        n_low_padj += 1/3

                elif policies['pricing_adjustment']['policy'] == 'by_inflation':
                    n_inf_padj += 1
                else:
                    raise ValueError(f"Uknown {policies['pricing_adjustment']['policy']}")
                n_padj +=1 

print("Participants used custom policies for these many times:")
print("- Budget allocation:")
print(f"    - By population: {n_pop_ba} • {n_pop_ba/n_ba*100:.1f}%")
print(f"    - By inverse population: {n_invpop_ba} • {n_invpop_ba/n_ba*100:.1f}%")
print(f"    - By income: {n_inc_ba} • {n_inc_ba/n_ba*100:.1f}%")
print(f"    - By inverse income: {n_invinc_ba} • {n_invinc_ba/n_ba*100:.1f}%")
print(f"    - Custom: {n_custom_ba} • {n_custom_ba/n_ba*100:.1f}%")
print()
print("- NRW mitigation:")
print(f"    - By population: {n_pop_nrw} • {n_pop_nrw/n_nrw*100:.1f}%")
print(f"    - By NRW class: {n_class_nrw} • {n_class_nrw/n_nrw*100:.1f}%")
print(f"    - Custom: {n_custom_nrw} • {n_custom_nrw/n_nrw*100:.1f}%")
print()
print("- Pricing adjustment:")
print(f"    - By inflation: {n_inf_padj} • {n_inf_padj/n_padj*100:.1f}%")
print(f"    - Custom: {n_custom_padj} • {n_custom_padj/n_padj*100:.1f}%")
print(f"         and the times below inflation where {n_low_padj/n_custom_padj*100:.1f}")

/Users/zannads/repos/water-futures-battle/visualization/.venv/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/Users/zannads/repos/water-futures-battle/visualization/.venv/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/Users/zannads/repos/water-futures-battle/visualization/.venv/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/Users/zannads/repos/water-futures-battle/visualization/.venv/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/Users/zannads/repos/water-futures-battle/visualization/.venv/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data 

Participants used custom policies for these many times:
- Budget allocation:
    - By population: 585 • 66.9%
    - By inverse population: 36 • 4.1%
    - By income: 27 • 3.1%
    - By inverse income: 10 • 1.1%
    - Custom: 217 • 24.8%

- NRW mitigation:
    - By population: 2553 • 29.2%
    - By NRW class: 5422 • 62.0%
    - Custom: 775 • 8.9%

- Pricing adjustment:
    - By inflation: 5028 • 57.5%
    - Custom: 3722 • 42.5%
         and the times below inflation where 71.6


In [7]:
import sys
project_root = Path("..").resolve()   # repo root when notebook is in visualization/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from scripts import bwf_rank

team_data_dict = bwf_rank.upload_teams_data(teams_dir=Path("..", "teams"))

n_teams_started = sum(1 for td in team_data_dict.values() if not td.reference)
n_teams_completed = sum(1 for td in team_data_dict.values() if not td.reference and td.complete)

rankings = bwf_rank.compute_rankings(team_data_dict)

bwf_rank.plot_scatter_perceptions(team_data_dict).show()